# 👁️ MPIIGaze 기반 눈 탐지 & 시선 추정 모델
## ALS 환우분을 위한 시선 추적 AAC 서비스

---

### 📋 전체 파이프라인
```
[원본 이미지] ──→ [YOLOv8-nano 눈 탐지] ──→ [눈 영역 크롭]
                                                    │
                                                    ↓
                                          [MobileNetV3 시선 추정]
                                                    │
                                                    ↓
                                          [화면 좌표 (x, y) 출력]
```

### 📁 MPIIGaze 데이터 구조
- `Data/Original/p{id}/day{d}/` — 원본 얼굴 이미지 (1280×720 JPG)
- `Data/Original/p{id}/day{d}/annotation.txt` — 눈 랜드마크 + 화면 시선좌표 (41개 필드)
- `Data/Normalized/p{id}/day{d}.mat` — 정규화된 눈 크롭(36×60) + 3D 시선벡터
- `Annotation Subset/p{id}.txt` — 간략 어노테이션 (15명 샘플)

### ⚙️ annotation.txt 41개 필드 구조
```
[0-11]  : 오른쪽 눈 6개 랜드마크 (x1,y1,...,x6,y6) — 이미지 픽셀 좌표
[12-23] : 왼쪽 눈 6개 랜드마크 (x1,y1,...,x6,y6) — 이미지 픽셀 좌표  
[24-25] : 화면 시선 목표점 (screen_x, screen_y) — 화면 픽셀 좌표
[26-28] : 머리 회전각 (degree)
[29-31] : 3D 시선 방향벡터 (yaw, pitch, roll in radians)
[32-40] : 3D 기하 변환 데이터
```

## 0️⃣ 환경 설정: 패키지 설치

> 처음 실행 시 한 번만 실행하면 됩니다.

In [ ]:
# 필요한 패키지 설치
# - ultralytics: YOLOv8 프레임워크
# - scipy: .mat 파일 파싱 (MATLAB 포맷)
# - mlflow: 실험 추적 및 모델 버전 관리
# - torchvision: MobileNetV3 등 사전학습 모델 포함
!pip install ultralytics scipy mlflow torchvision tqdm Pillow matplotlib seaborn --quiet

print('✅ 패키지 설치 완료')

## 1️⃣ 라이브러리 임포트 & 하이퍼파라미터 설정

> **여기서 모든 학습 설정을 바꿀 수 있습니다.** 실험을 반복할 때 이 셀만 수정하세요.

In [ ]:
import os
import math
import glob
import shutil
import random
import zipfile
from pathlib import Path

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models

import mlflow
import mlflow.pytorch

# ─────────────────────────────────────────────────────────────
#  📁 경로 설정
# ─────────────────────────────────────────────────────────────
BASE_DIR     = Path('C:/Users/SSAFY/Desktop/GAZE-CAPTURE/GazeCapture/MPIIGAZE')
DATA_DIR     = BASE_DIR / 'MPIIGaze' / 'Data'
ANNOT_DIR    = BASE_DIR / 'MPIIGaze' / 'Annotation Subset'
YOLO_DIR     = BASE_DIR / 'yolo_dataset'   # YOLO 학습용 데이터셋 저장 경로
RUNS_DIR     = BASE_DIR / 'runs'            # 실험 결과 저장 경로 (exp1, exp2...)
MLFLOW_DIR   = BASE_DIR / 'mlruns'          # MLflow 로그 저장 경로

# ─────────────────────────────────────────────────────────────
#  🔧 YOLO 학습 하이퍼파라미터
# ─────────────────────────────────────────────────────────────
YOLO_EPOCHS      = 30        # YOLO 학습 에포크 수 (눈 탐지 모델)
YOLO_IMGSZ       = 640       # YOLO 입력 이미지 크기 (정사각형 변환)
YOLO_BATCH       = 16        # YOLO 배치 크기
YOLO_MODEL       = 'yolov8n.pt'  # nano 모델 — 웹 배포용 경량 모델
EYE_PADDING      = 0.35      # 눈 바운딩박스 패딩 비율 (랜드마크 → 박스 변환 시)
TRAIN_SPLIT      = 0.8       # 학습/검증 비율 (8:2)

# ─────────────────────────────────────────────────────────────
#  🧠 시선 추정 모델 하이퍼파라미터
# ─────────────────────────────────────────────────────────────
GAZE_EPOCHS      = 50        # 시선 추정 학습 에포크 수
GAZE_BATCH       = 128       # 배치 크기 (눈 크롭 이미지가 작으므로 크게 설정)
GAZE_LR          = 1e-3      # 초기 학습률 (Adam optimizer)
GAZE_LR_STEP     = 15        # 학습률 감소 주기 (에포크 단위)
GAZE_LR_GAMMA    = 0.5       # 학습률 감소 비율 (STEP마다 LR × GAMMA)
GAZE_INPUT_SIZE  = 64        # 눈 크롭 이미지 리사이즈 크기 (64×64)
GAZE_OUTPUT_DIM  = 2         # 출력 차원: (yaw, pitch) — 시선 방향 각도 2개
GAZE_WEIGHT_DECAY = 1e-4     # L2 정규화 (과적합 방지)

# ─────────────────────────────────────────────────────────────
#  🖥️ 디바이스 설정
# ─────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'📁 BASE_DIR   : {BASE_DIR}')
print(f'📁 DATA_DIR   : {DATA_DIR}')
print(f'🖥️  Device     : {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU        : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# 재현성을 위한 시드 고정
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(42)

print('\n✅ 설정 완료')

## 2️⃣ 데이터 탐색 (EDA)

데이터셋의 구조와 샘플을 먼저 시각적으로 확인합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  데이터셋 통계 파악
# ─────────────────────────────────────────────────────────────

def count_dataset_stats(data_dir: Path) -> dict:
    """데이터셋 전체 통계를 계산합니다."""
    stats = {'persons': [], 'total_images': 0, 'total_days': 0}
    
    original_dir = data_dir / 'Original'
    for person_dir in sorted(original_dir.iterdir()):
        if not person_dir.is_dir():
            continue
        person_id = person_dir.name  # p00, p01, ...
        n_days = len(list(person_dir.iterdir()))
        
        # 이미지 개수 세기
        n_images = len(list(person_dir.glob('**/*.jpg')))
        stats['persons'].append({'id': person_id, 'days': n_days, 'images': n_images})
        stats['total_images'] += n_images
        stats['total_days'] += n_days
    
    return stats

print('📊 데이터셋 통계 계산 중...')
stats = count_dataset_stats(DATA_DIR)

print(f'\n총 피험자 수: {len(stats["persons"])}명')
print(f'총 수집 일수: {stats["total_days"]}일')
print(f'총 이미지 수: {stats["total_images"]:,}장')
print(f'\n피험자별 데이터:')
for p in stats['persons']:
    print(f'  {p["id"]}: {p["days"]}일 세션, {p["images"]:,}장')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  원본 이미지 + 눈 랜드마크 시각화
# ─────────────────────────────────────────────────────────────

def load_annotation(ann_path: Path):
    """annotation.txt 파일 파싱.
    
    반환값 (각 항목은 numpy array):
        right_eye_pts : (6, 2) 오른쪽 눈 랜드마크 픽셀 좌표
        left_eye_pts  : (6, 2) 왼쪽 눈 랜드마크 픽셀 좌표
        screen_gaze   : (2,)   화면 위 시선 목표 좌표
        gaze_vec      : (3,)   3D 시선 방향 벡터 (yaw, pitch, roll in radians)
    """
    annotations = []
    with open(ann_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 41:
                continue
            vals = [float(x) for x in parts]
            # 오른쪽 눈: 필드 0-11 → (6, 2) 좌표 배열
            right_pts = np.array(vals[0:12]).reshape(6, 2)
            # 왼쪽 눈: 필드 12-23 → (6, 2) 좌표 배열
            left_pts  = np.array(vals[12:24]).reshape(6, 2)
            # 화면 시선 목표점: 필드 24-25
            screen_gaze = np.array(vals[24:26])
            # 3D 시선 벡터: 필드 29-31 (yaw, pitch, roll)
            gaze_vec = np.array(vals[29:32])
            
            annotations.append({
                'right_pts': right_pts,
                'left_pts': left_pts,
                'screen_gaze': screen_gaze,
                'gaze_vec': gaze_vec
            })
    return annotations


def landmarks_to_bbox(pts: np.ndarray, padding: float, img_w: int, img_h: int):
    """눈 랜드마크 포인트 집합 → 패딩 포함 바운딩박스 (픽셀 좌표).
    
    Args:
        pts     : (N, 2) 랜드마크 좌표
        padding : 박스 크기 대비 여백 비율
        img_w/h : 이미지 크기 (경계 클리핑용)
    Returns:
        (x1, y1, x2, y2) 픽셀 좌표
    """
    x1, y1 = pts[:, 0].min(), pts[:, 1].min()
    x2, y2 = pts[:, 0].max(), pts[:, 1].max()
    
    # 눈 크기 기준 패딩 추가 (눈 주변 맥락 정보 포함)
    pw = (x2 - x1) * padding
    ph = (y2 - y1) * padding
    
    # 이미지 경계를 벗어나지 않도록 클리핑
    x1 = max(0, x1 - pw)
    y1 = max(0, y1 - ph)
    x2 = min(img_w, x2 + pw)
    y2 = min(img_h, y2 + ph)
    
    return x1, y1, x2, y2


# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

person_dirs = sorted((DATA_DIR / 'Original').iterdir())

for idx, person_dir in enumerate(person_dirs[:6]):
    # 첫 번째 day의 첫 번째 이미지 로드
    day_dirs = sorted(person_dir.iterdir())
    ann_file = day_dirs[0] / 'annotation.txt'
    
    # 이미지 파일 찾기 (annotation 순서대로 번호가 매겨짐)
    img_file = sorted(day_dirs[0].glob('*.jpg'))[0]
    
    img = np.array(Image.open(img_file))
    anns = load_annotation(ann_file)
    
    if not anns:
        continue
    
    ann = anns[0]  # 첫 번째 프레임 어노테이션
    h, w = img.shape[:2]
    
    # 바운딩박스 계산
    rx1, ry1, rx2, ry2 = landmarks_to_bbox(ann['right_pts'], EYE_PADDING, w, h)
    lx1, ly1, lx2, ly2 = landmarks_to_bbox(ann['left_pts'], EYE_PADDING, w, h)
    
    ax = axes[idx]
    ax.imshow(img)
    
    # 오른쪽 눈 박스 (파란색)
    ax.add_patch(patches.Rectangle(
        (rx1, ry1), rx2-rx1, ry2-ry1,
        linewidth=2, edgecolor='blue', facecolor='none', label='Right eye'
    ))
    # 왼쪽 눈 박스 (빨간색)
    ax.add_patch(patches.Rectangle(
        (lx1, ly1), lx2-lx1, ly2-ly1,
        linewidth=2, edgecolor='red', facecolor='none', label='Left eye'
    ))
    
    # 랜드마크 포인트 표시
    ax.scatter(ann['right_pts'][:, 0], ann['right_pts'][:, 1],
               c='cyan', s=20, zorder=5)
    ax.scatter(ann['left_pts'][:, 0], ann['left_pts'][:, 1],
               c='yellow', s=20, zorder=5)
    
    ax.set_title(f'{person_dir.name} | gaze=({ann["screen_gaze"][0]:.0f}, {ann["screen_gaze"][1]:.0f})',
                 fontsize=10)
    ax.axis('off')

# 범례
from matplotlib.lines import Line2D
legend_elements = [
    patches.Patch(facecolor='none', edgecolor='blue', label='Right eye box'),
    patches.Patch(facecolor='none', edgecolor='red', label='Left eye box'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='cyan', label='Right landmarks'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='yellow', label='Left landmarks'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=11)
plt.suptitle('MPIIGaze 원본 이미지 + 눈 랜드마크', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'eda_samples.png'), dpi=100, bbox_inches='tight')
plt.show()
print('✅ EDA 시각화 저장: eda_samples.png')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Normalized .mat 파일 탐색 — 시선 추정 학습 데이터
# ─────────────────────────────────────────────────────────────

def load_mat_file(mat_path: Path):
    """MPIIGaze .mat 파일 파싱.
    
    .mat 파일 구조:
        data['right']['gaze']  : (N, 3)  오른쪽 눈 3D 시선 방향벡터
        data['right']['image'] : (N, 36, 60)  오른쪽 눈 크롭 (회색조 36×60)
        data['right']['pose']  : (N, 3)  머리 자세 벡터
        data['left'] ...       : 왼쪽 눈 (동일 구조)
        filenames              : (N, 1)  대응 이미지 파일명
    
    Returns:
        list of dict: {'image': (36,60) uint8, 'gaze': (3,) float32, 'side': 'right'/'left'}
    """
    mat = sio.loadmat(str(mat_path))
    data = mat['data']
    samples = []
    
    for side in ['right', 'left']:
        side_data = data[0, 0][side][0, 0]
        # gaze: (N, 3) 시선 방향벡터 (camera space unit vector)
        gazes  = side_data['gaze'].astype(np.float32)   # (N, 3)
        # image: (N, 36, 60) 회색조 눈 크롭 이미지
        images = side_data['image']                     # (N, 36, 60) uint8
        
        for i in range(len(gazes)):
            samples.append({
                'image': images[i],   # (36, 60) numpy array
                'gaze': gazes[i],     # (3,) float32
                'side': side
            })
    
    return samples


def gaze_vector_to_angles(gaze_vec: np.ndarray):
    """3D 시선벡터 → (yaw, pitch) 각도 변환 (라디안).
    
    yaw  : 좌우 시선 방향 (-π/2 ~ +π/2)
    pitch: 상하 시선 방향 (-π/2 ~ +π/2)
    
    Web 배포 시 실제 화면 좌표 변환:
        screen_x = screen_center_x + tan(yaw) * focal_length
        screen_y = screen_center_y + tan(pitch) * focal_length
    """
    gx, gy, gz = gaze_vec
    # yaw: x축 기준 좌우 각도
    yaw   = math.atan2(gx, -gz)                     # 좌우 (-값: 왼쪽 시선)
    # pitch: y축 기준 상하 각도
    pitch = math.atan2(-gy, math.sqrt(gx**2 + gz**2))  # 상하 (-값: 아래 시선)
    return yaw, pitch


# 샘플 .mat 파일 로드 및 시각화
sample_mat = DATA_DIR / 'Normalized' / 'p00' / 'day01.mat'
samples = load_mat_file(sample_mat)

print(f'샘플 .mat 파일: {sample_mat.name}')
print(f'총 샘플 수: {len(samples)} (양쪽 눈)')
print(f'이미지 크기: {samples[0]["image"].shape}  (H=36, W=60)')
print(f'시선벡터 예시: {samples[0]["gaze"]}  (3D 단위벡터)')

yaw, pitch = gaze_vector_to_angles(samples[0]['gaze'])
print(f'→ yaw={math.degrees(yaw):.1f}°, pitch={math.degrees(pitch):.1f}°')

# 눈 크롭 이미지 시각화
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
axes = axes.flatten()
for i, ax in enumerate(axes):
    s = samples[i]
    yaw, pitch = gaze_vector_to_angles(s['gaze'])
    ax.imshow(s['image'], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"{s['side'][0]}:\n{math.degrees(yaw):.0f}°,{math.degrees(pitch):.0f}°",
                 fontsize=7)
    ax.axis('off')

plt.suptitle('정규화된 눈 크롭 이미지 (36×60) + 시선 각도', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(BASE_DIR / 'eda_eye_crops.png'), dpi=100, bbox_inches='tight')
plt.show()

# 시선 분포 히스토그램
all_yaws, all_pitches = [], []
for s in samples:
    yaw, pitch = gaze_vector_to_angles(s['gaze'])
    all_yaws.append(math.degrees(yaw))
    all_pitches.append(math.degrees(pitch))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_yaws, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Yaw (°, 좌우)', fontsize=12)
axes[0].set_ylabel('빈도', fontsize=12)
axes[0].set_title('시선 좌우 분포', fontsize=12)

axes[1].hist(all_pitches, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Pitch (°, 상하)', fontsize=12)
axes[1].set_ylabel('빈도', fontsize=12)
axes[1].set_title('시선 상하 분포', fontsize=12)

plt.tight_layout()
plt.savefig(str(BASE_DIR / 'eda_gaze_dist.png'), dpi=100, bbox_inches='tight')
plt.show()
print('✅ 탐색 완료')

## 3️⃣ YOLO 학습용 데이터셋 생성

`annotation.txt`의 눈 랜드마크 좌표 → YOLO 포맷 `.txt` 라벨로 변환

**YOLO 라벨 포맷:** `<class_id> <cx> <cy> <w> <h>` (모두 0~1로 정규화)
- Class 0: `right_eye` (피험자 기준 오른쪽 눈 = 이미지 왼편)
- Class 1: `left_eye`  (피험자 기준 왼쪽 눈 = 이미지 오른편)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  YOLO 데이터셋 디렉토리 구조 생성
# ─────────────────────────────────────────────────────────────
#
#  yolo_dataset/
#  ├── images/
#  │   ├── train/   ← 80% 학습 이미지 (심볼릭 링크 or 복사)
#  │   └── val/     ← 20% 검증 이미지
#  ├── labels/
#  │   ├── train/   ← 대응 YOLO 라벨 .txt
#  │   └── val/
#  └── dataset.yaml ← YOLOv8 설정 파일

for split in ['train', 'val']:
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print('📂 YOLO 디렉토리 구조 생성 완료')
print(f'  {YOLO_DIR}/')
print('  ├── images/train/')
print('  ├── images/val/')
print('  ├── labels/train/')
print('  └── labels/val/')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  annotation.txt → YOLO 라벨 변환 함수
# ─────────────────────────────────────────────────────────────

def create_yolo_label(ann: dict, img_w: int, img_h: int,
                       padding: float = EYE_PADDING) -> str:
    """하나의 어노테이션 → YOLO 라벨 문자열 생성.
    
    Returns:
        '0 cx cy w h\n1 cx cy w h' 형식의 라벨 문자열
        (오른쪽 눈: class 0, 왼쪽 눈: class 1)
    """
    lines = []
    
    for class_id, pts in [(0, ann['right_pts']), (1, ann['left_pts'])]:
        # 랜드마크 → 바운딩박스 (패딩 포함)
        x1, y1, x2, y2 = landmarks_to_bbox(pts, padding, img_w, img_h)
        
        # 픽셀 좌표 → YOLO 정규화 좌표 (0~1)
        cx = (x1 + x2) / 2 / img_w
        cy = (y1 + y2) / 2 / img_h
        bw = (x2 - x1) / img_w
        bh = (y2 - y1) / img_h
        
        # 박스가 유효한 경우만 추가 (최소 크기 필터)
        if bw > 0.005 and bh > 0.005:
            lines.append(f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    
    return '\n'.join(lines)


def process_day_folder(day_dir: Path, ann_file: Path,
                        img_size=(1280, 720)):
    """하루치 annotation.txt → (이미지경로, 라벨문자열) 쌍 반환."""
    anns = load_annotation(ann_file)
    # 이미지 파일 정렬 (0001.jpg, 0002.jpg, ...)
    img_files = sorted(day_dir.glob('*.jpg'))
    
    pairs = []
    for img_file, ann in zip(img_files, anns):
        label_str = create_yolo_label(ann, img_size[0], img_size[1])
        if label_str.strip():  # 라벨이 비어있지 않은 경우만
            pairs.append((img_file, label_str))
    
    return pairs


print('🔧 변환 함수 정의 완료')
print('  - create_yolo_label(): annotation → YOLO .txt')
print('  - process_day_folder(): 하루치 폴더 일괄 처리')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  전체 데이터셋 변환 + 8:2 분할
# ─────────────────────────────────────────────────────────────

# 전체 (이미지경로, 라벨) 쌍 수집
all_pairs = []

print('📦 어노테이션 변환 시작...')
person_dirs = sorted((DATA_DIR / 'Original').iterdir())

for person_dir in tqdm(person_dirs, desc='Persons'):
    if not person_dir.is_dir():
        continue
    
    for day_dir in sorted(person_dir.iterdir()):
        if not day_dir.is_dir():
            continue
        
        ann_file = day_dir / 'annotation.txt'
        if not ann_file.exists():
            continue
        
        pairs = process_day_folder(day_dir, ann_file)
        all_pairs.extend(pairs)

print(f'\n총 유효 샘플 수: {len(all_pairs):,}장')

# 무작위 섞기 후 8:2 분할
random.shuffle(all_pairs)
n_train = int(len(all_pairs) * TRAIN_SPLIT)
train_pairs = all_pairs[:n_train]
val_pairs   = all_pairs[n_train:]

print(f'학습 세트: {len(train_pairs):,}장')
print(f'검증 세트: {len(val_pairs):,}장')
print(f'비율: {len(train_pairs)/len(all_pairs)*100:.0f}% : {len(val_pairs)/len(all_pairs)*100:.0f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  이미지 복사 + 라벨 파일 저장
# ─────────────────────────────────────────────────────────────
#
# 주의: 이미지를 물리적으로 복사하면 디스크를 많이 사용합니다.
# YOLO 학습 시 images/ 경로와 labels/ 경로를 매핑해서 사용합니다.

def save_yolo_split(pairs: list, split: str):
    """이미지 + 라벨을 YOLO 디렉토리로 저장."""
    img_dir = YOLO_DIR / 'images' / split
    lbl_dir = YOLO_DIR / 'labels' / split
    
    saved = 0
    skipped = 0
    
    for img_path, label_str in tqdm(pairs, desc=f'Saving {split}'):
        # 고유 파일명 생성 (person_day_frame 조합)
        # 예: p00_day01_0001.jpg
        parts = img_path.parts
        # parts 구조: [..., 'Original', 'p00', 'day01', '0001.jpg']
        try:
            orig_idx = [p for p in parts].index('Original')
            person  = parts[orig_idx + 1]
            day     = parts[orig_idx + 2]
            frame   = img_path.stem
            unique_name = f'{person}_{day}_{frame}'
        except (ValueError, IndexError):
            unique_name = img_path.stem
        
        dst_img = img_dir / f'{unique_name}.jpg'
        dst_lbl = lbl_dir / f'{unique_name}.txt'
        
        # 이미지 복사 (이미 존재하면 스킵)
        if not dst_img.exists():
            shutil.copy2(img_path, dst_img)
        
        # 라벨 저장
        dst_lbl.write_text(label_str)
        saved += 1
    
    return saved

print('📁 학습 데이터 저장 중...')
n_train_saved = save_yolo_split(train_pairs, 'train')
n_val_saved   = save_yolo_split(val_pairs, 'val')

print(f'\n✅ 저장 완료')
print(f'  학습: {n_train_saved:,}장 → {YOLO_DIR}/images/train/')
print(f'  검증: {n_val_saved:,}장 → {YOLO_DIR}/images/val/')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  dataset.yaml 생성 — YOLOv8 설정 파일
# ─────────────────────────────────────────────────────────────

yaml_content = f"""# MPIIGaze 눈 탐지 데이터셋 설정
# YOLOv8 학습 시 이 파일을 참조합니다.

# 데이터 경로 (절대 경로 사용)
path: {str(YOLO_DIR).replace(chr(92), '/')}
train: images/train
val: images/val

# 클래스 수와 이름
nc: 2
names:
  0: right_eye   # 피험자 기준 오른쪽 눈 (이미지 왼편)
  1: left_eye    # 피험자 기준 왼쪽 눈 (이미지 오른편)
"""

yaml_path = YOLO_DIR / 'dataset.yaml'
yaml_path.write_text(yaml_content, encoding='utf-8')

print(f'✅ dataset.yaml 생성: {yaml_path}')
print('\n내용:')
print(yaml_content)

## 4️⃣ YOLOv8-nano 눈 탐지 모델 학습

사전학습된 `yolov8n.pt`를 MPIIGaze 눈 데이터로 파인튜닝합니다.

> **왜 nano 모델인가?**  
> 웹 브라우저에서 ONNX Runtime Web으로 실행해야 하므로 모델이 작아야 합니다.  
> YOLOv8n은 ~3.2MB로 브라우저에서도 충분히 빠릅니다.

In [ ]:
from ultralytics import YOLO

# ─────────────────────────────────────────────────────────────
#  실험 폴더 자동 증가 생성 (exp1, exp2, exp3 ...)
# ─────────────────────────────────────────────────────────────

def get_next_exp_dir(runs_dir: Path, prefix: str = 'exp') -> Path:
    """다음 실험 번호를 자동으로 찾아 폴더를 생성합니다."""
    idx = 1
    while (runs_dir / f'{prefix}{idx}').exists():
        idx += 1
    exp_dir = runs_dir / f'{prefix}{idx}'
    exp_dir.mkdir(parents=True, exist_ok=True)
    return exp_dir

YOLO_EXP_DIR = get_next_exp_dir(RUNS_DIR / 'yolo')
print(f'📁 YOLO 실험 경로: {YOLO_EXP_DIR}')

# ─────────────────────────────────────────────────────────────
#  YOLOv8 학습 시작
# ─────────────────────────────────────────────────────────────

# 사전학습된 YOLOv8-nano 모델 로드 (없으면 자동 다운로드)
yolo_model = YOLO(YOLO_MODEL)

print(f'\n🚀 YOLOv8 학습 시작')
print(f'   모델: {YOLO_MODEL}')
print(f'   에포크: {YOLO_EPOCHS}')
print(f'   배치: {YOLO_BATCH}')
print(f'   이미지 크기: {YOLO_IMGSZ}×{YOLO_IMGSZ}')

results = yolo_model.train(
    data      = str(YOLO_DIR / 'dataset.yaml'),
    epochs    = YOLO_EPOCHS,
    imgsz     = YOLO_IMGSZ,
    batch     = YOLO_BATCH,
    device    = 0 if DEVICE == 'cuda' else 'cpu',
    project   = str(RUNS_DIR / 'yolo'),  # 저장 루트 경로
    name      = YOLO_EXP_DIR.name,        # 실험 폴더 이름 (exp1, exp2...)
    exist_ok  = True,
    
    # 하이퍼파라미터
    lr0       = 0.01,     # 초기 학습률
    lrf       = 0.01,     # 최종 학습률 비율 (lr0 × lrf)
    momentum  = 0.937,
    weight_decay = 0.0005,
    
    # 데이터 증강 설정 (눈 이미지는 뒤집으면 안 됨)
    flipud    = 0.0,      # 상하 뒤집기 없음
    fliplr    = 0.5,      # 좌우 뒤집기 50% (단, class 교체 처리 포함)
    mosaic    = 1.0,      # 모자이크 증강
    
    # 저장 설정
    save      = True,
    save_period = 5,      # 5에포크마다 체크포인트 저장
    plots     = True,     # 학습 곡선 자동 저장
    verbose   = True,
)

print(f'\n✅ YOLO 학습 완료!')
print(f'   가중치 저장 위치: {YOLO_EXP_DIR}/weights/')
print(f'   best.pt: 검증 mAP 기준 최고 모델')
print(f'   last.pt: 마지막 에포크 모델')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  YOLO 검증 및 추론 테스트
# ─────────────────────────────────────────────────────────────

# 최고 모델로 검증 세트 평가
best_yolo_weights = YOLO_EXP_DIR / 'weights' / 'best.pt'
if best_yolo_weights.exists():
    yolo_best = YOLO(str(best_yolo_weights))
    val_results = yolo_best.val(data=str(YOLO_DIR / 'dataset.yaml'))
    print(f'\n📊 YOLO 검증 결과:')
    print(f'   mAP50:   {val_results.box.map50:.4f}')
    print(f'   mAP50-95: {val_results.box.map:.4f}')
    print(f'   Precision: {val_results.box.mp:.4f}')
    print(f'   Recall:    {val_results.box.mr:.4f}')

# 샘플 이미지로 추론 테스트
sample_imgs = list((YOLO_DIR / 'images' / 'val').glob('*.jpg'))[:4]
if sample_imgs and best_yolo_weights.exists():
    pred_results = yolo_best.predict(
        source=sample_imgs,
        conf=0.5,
        iou=0.45,
        save=False
    )
    
    fig, axes = plt.subplots(1, min(4, len(pred_results)), figsize=(16, 4))
    if len(pred_results) == 1:
        axes = [axes]
    
    for ax, res in zip(axes, pred_results):
        # 결과 이미지 (박스 + 클래스명 + 신뢰도 그려진 numpy array)
        img_with_boxes = res.plot()
        ax.imshow(img_with_boxes[:, :, ::-1])  # BGR → RGB 변환
        ax.axis('off')
    
    plt.suptitle('YOLO 눈 탐지 추론 결과', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'yolo_predictions.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print('✅ 추론 결과 저장: yolo_predictions.png')

## 5️⃣ 시선 추정 PyTorch 커스텀 데이터셋

`.mat` 파일에서 눈 크롭 이미지(36×60)와 시선 벡터를 불러옵니다.

**입력:** 눈 이미지 (64×64 RGB로 리사이즈)  
**출력:** 시선 각도 (yaw, pitch) — 단위: 라디안

In [ ]:
class MPIIGazeDataset(Dataset):
    """MPIIGaze Normalized 데이터셋 — 시선 회귀 학습용.
    
    .mat 파일에서 (눈 이미지, 시선 각도) 쌍을 로드합니다.
    양쪽 눈 데이터를 모두 활용하여 데이터 양을 2배로 늘립니다.
    
    __getitem__ 반환:
        image : Tensor (3, H, W), float32, 0~1 정규화
        label : Tensor (2,), float32  — (yaw, pitch) in radians
    """
    
    def __init__(self,
                 normalized_dir: Path,
                 input_size: int = GAZE_INPUT_SIZE,
                 augment: bool = True):
        """
        Args:
            normalized_dir: Data/Normalized 경로
            input_size    : 모델 입력 이미지 크기 (정사각형)
            augment       : True이면 학습용 증강 적용
        """
        self.input_size = input_size
        self.augment    = augment
        self.samples    = []  # List of (image_array, gaze_angles)
        
        # 전처리 파이프라인
        # 회색조 36×60 → 3채널 64×64 RGB → 정규화
        if augment:
            self.transform = T.Compose([
                T.ToPILImage(),
                T.Resize((input_size, input_size)),
                # 학습 증강: 밝기/대비 랜덤 변화 (눈의 조명 변화 모사)
                T.ColorJitter(brightness=0.3, contrast=0.3),
                # 가우시안 블러 (카메라 흔들림 모사)
                T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.3),
                T.ToTensor(),  # (H, W, C) uint8 → (C, H, W) float32, 0~1
                # ImageNet 통계로 정규화 (MobileNetV3 사전학습 기준)
                T.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225]),
            ])
        else:
            self.transform = T.Compose([
                T.ToPILImage(),
                T.Resize((input_size, input_size)),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225]),
            ])
        
        # .mat 파일 전체 로드
        mat_files = sorted(normalized_dir.glob('**/*.mat'))
        print(f'  .mat 파일 수: {len(mat_files)}')
        
        for mat_path in tqdm(mat_files, desc='Loading .mat files'):
            try:
                mat = sio.loadmat(str(mat_path))
                data = mat['data']
                
                for side in ['right', 'left']:
                    side_data = data[0, 0][side][0, 0]
                    gazes  = side_data['gaze'].astype(np.float32)  # (N, 3)
                    images = side_data['image']                    # (N, 36, 60) uint8
                    
                    for i in range(len(gazes)):
                        # 3D 시선벡터 → (yaw, pitch) 각도 변환
                        gx, gy, gz = gazes[i]
                        yaw   = math.atan2(gx, -gz)
                        pitch = math.atan2(-gy, math.sqrt(gx**2 + gz**2))
                        
                        # 왼쪽 눈의 경우 yaw를 수평 반전
                        # (왼눈과 오른눈은 대칭 관계이므로 통합 학습 가능)
                        if side == 'left':
                            yaw = -yaw
                        
                        self.samples.append((
                            images[i].copy(),              # (36, 60) uint8
                            np.array([yaw, pitch], dtype=np.float32)  # (2,)
                        ))
            except Exception as e:
                print(f'  ⚠️ {mat_path.name} 로드 실패: {e}')
        
        print(f'  ✅ 총 {len(self.samples):,}개 샘플 로드 완료')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_arr, gaze_angles = self.samples[idx]
        
        # 회색조 (36, 60) → RGB (36, 60, 3) 변환
        # MobileNetV3는 3채널 입력을 요구하므로 채널 복제
        img_rgb = np.stack([img_arr, img_arr, img_arr], axis=-1)  # (36, 60, 3)
        
        # 전처리 파이프라인 적용 (리사이즈 + 정규화)
        img_tensor = self.transform(img_rgb)  # (3, 64, 64)
        
        # gaze_angles를 PyTorch Tensor로 변환
        label = torch.tensor(gaze_angles, dtype=torch.float32)  # (2,)
        
        return img_tensor, label


print('📋 MPIIGazeDataset 클래스 정의 완료')
print('  입력: 눈 크롭 이미지 (36×60 grayscale → 64×64 RGB Tensor)')
print('  출력: [yaw, pitch] 라디안 (float32)')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  데이터셋 로드 + DataLoader 생성
# ─────────────────────────────────────────────────────────────

print('📦 데이터셋 로딩...')
full_dataset = MPIIGazeDataset(
    normalized_dir = DATA_DIR / 'Normalized',
    input_size     = GAZE_INPUT_SIZE,
    augment        = True  # 증강 포함 (학습/검증 분리 후 검증엔 False로 재설정)
)

# 8:2 분할
n_total = len(full_dataset)
n_train = int(n_total * TRAIN_SPLIT)
n_val   = n_total - n_train

train_ds, val_ds = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

# 검증 세트는 증강 비활성화
# (검증 세트의 transform을 바꾸기 위해 별도 데이터셋 생성)
val_dataset = MPIIGazeDataset(
    normalized_dir = DATA_DIR / 'Normalized',
    input_size     = GAZE_INPUT_SIZE,
    augment        = False
)
_, val_ds_clean = random_split(
    val_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

# DataLoader 생성
# num_workers: 병렬 데이터 로딩 프로세스 수 (Windows에서는 0 권장)
train_loader = DataLoader(
    train_ds,
    batch_size  = GAZE_BATCH,
    shuffle     = True,
    num_workers = 0,
    pin_memory  = True if DEVICE == 'cuda' else False
)

val_loader = DataLoader(
    val_ds_clean,
    batch_size  = GAZE_BATCH,
    shuffle     = False,
    num_workers = 0,
    pin_memory  = True if DEVICE == 'cuda' else False
)

print(f'\n학습 데이터: {n_train:,}개')
print(f'검증 데이터: {n_val:,}개')
print(f'학습 배치 수: {len(train_loader)}')
print(f'검증 배치 수: {len(val_loader)}')

# 배치 샘플 확인
sample_batch = next(iter(train_loader))
print(f'\n배치 확인:')
print(f'  이미지 shape: {sample_batch[0].shape}  (B, C, H, W)')
print(f'  레이블 shape: {sample_batch[1].shape}  (B, 2) — [yaw, pitch]')
print(f'  레이블 범위: yaw=[{sample_batch[1][:,0].min():.2f}, {sample_batch[1][:,0].max():.2f}]')
print(f'               pitch=[{sample_batch[1][:,1].min():.2f}, {sample_batch[1][:,1].max():.2f}]')

## 6️⃣ MobileNetV3-Small 시선 추정 모델

**왜 MobileNetV3-Small?**
- 파라미터 수 ~2.5M으로 매우 경량 (ResNet50 대비 1/10)
- ImageNet 사전학습 가중치로 초기화 → 빠른 수렴
- ONNX 변환 후 브라우저에서 `onnxruntime-web`으로 직접 실행 가능

In [ ]:
class GazeEstimator(nn.Module):
    """MobileNetV3-Small 기반 시선 추정 모델.
    
    구조:
        MobileNetV3-Small backbone (사전학습)
        → Dropout(0.3)  ← 과적합 방지
        → Linear(576, 128)  ← 특징 압축
        → ReLU
        → Dropout(0.2)
        → Linear(128, 2)  ← (yaw, pitch) 출력
    
    입력: (B, 3, 64, 64) float32 Tensor
    출력: (B, 2) float32 Tensor — [yaw, pitch] in radians
    """
    
    def __init__(self, output_dim: int = GAZE_OUTPUT_DIM, pretrained: bool = True):
        super().__init__()
        
        # ── Backbone: MobileNetV3-Small ──────────────────────
        # 사전학습된 ImageNet 가중치 사용 (None이면 랜덤 초기화)
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.mobilenet_v3_small(weights=weights)
        
        # 원래 분류기(Classifier)를 제거하고 특징 추출기만 사용
        # backbone.features: Conv+BN+ReLU 레이어들
        # backbone.avgpool: 글로벌 평균 풀링 (출력: B × 576 × 1 × 1)
        self.features = backbone.features  # 특징 추출
        self.avgpool  = backbone.avgpool   # Global Average Pooling
        
        # ── Regression Head: 회귀 출력층 ─────────────────────
        # MobileNetV3-Small의 마지막 특징 채널: 576
        in_features = 576
        self.regressor = nn.Sequential(
            nn.Dropout(p=0.3),                    # 과적합 방지
            nn.Linear(in_features, 128),           # 특징 압축
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(128, output_dim),            # (yaw, pitch) 출력
        )
        
        # 회귀 헤드 가중치 초기화 (Xavier 균등 분포)
        for layer in self.regressor:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        # x: (B, 3, 64, 64)
        x = self.features(x)    # → (B, 576, H', W')
        x = self.avgpool(x)     # → (B, 576, 1, 1)
        x = x.flatten(1)        # → (B, 576)
        x = self.regressor(x)   # → (B, 2)  [yaw, pitch]
        return x
    
    def count_parameters(self):
        """학습 가능한 파라미터 수 반환."""
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


# 모델 생성 및 GPU 이동
gaze_model = GazeEstimator(output_dim=GAZE_OUTPUT_DIM, pretrained=True).to(DEVICE)

total_params, trainable_params = gaze_model.count_parameters()
print('🧠 GazeEstimator (MobileNetV3-Small):')
print(f'  전체 파라미터:       {total_params:,}')
print(f'  학습 가능 파라미터: {trainable_params:,}')
print(f'  모델 크기 추정:      ~{total_params * 4 / 1e6:.1f} MB (float32 기준)')
print(f'  Device: {DEVICE}')

# 더미 입력으로 순전파 테스트
dummy_input = torch.randn(4, 3, GAZE_INPUT_SIZE, GAZE_INPUT_SIZE).to(DEVICE)
with torch.no_grad():
    dummy_output = gaze_model(dummy_input)
print(f'  순전파 테스트: 입력 {tuple(dummy_input.shape)} → 출력 {tuple(dummy_output.shape)} ✅')

## 7️⃣ 학습 루프 + MLflow 실험 관리

**손실 함수:**
- `SmoothL1Loss` (=Huber Loss): MSE와 MAE의 장점을 결합
  - 큰 오차: MAE처럼 완만 → 이상치 robust
  - 작은 오차: MSE처럼 정밀 → 세밀한 수렴
- 각도 오차 지표: **Mean Angular Error (MAE, 단위: °)**

In [ ]:
# ─────────────────────────────────────────────────────────────
#  손실 함수 & 옵티마이저 & 스케줄러 설정
# ─────────────────────────────────────────────────────────────

# SmoothL1Loss (Huber Loss): 시선 각도 회귀에 효과적
# beta=1.0 → |error| < 1 이면 MSE, 그 이상이면 MAE
criterion = nn.SmoothL1Loss(reduction='mean', beta=0.1)

# Adam 옵티마이저: 학습률 자동 조절, 실제 프로젝트에서 가장 범용적
optimizer = optim.Adam(
    gaze_model.parameters(),
    lr           = GAZE_LR,
    weight_decay = GAZE_WEIGHT_DECAY  # L2 정규화
)

# StepLR 스케줄러: GAZE_LR_STEP 에포크마다 학습률을 GAZE_LR_GAMMA 배 감소
# 예: LR=0.001, step=15, gamma=0.5 → 15에포크: 0.0005, 30에포크: 0.00025
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size = GAZE_LR_STEP,
    gamma     = GAZE_LR_GAMMA
)

print('⚙️ 최적화 설정:')
print(f'  손실 함수: SmoothL1Loss (beta=0.1)')
print(f'  옵티마이저: Adam (lr={GAZE_LR}, weight_decay={GAZE_WEIGHT_DECAY})')
print(f'  스케줄러: StepLR (step={GAZE_LR_STEP}, gamma={GAZE_LR_GAMMA})')


# ─────────────────────────────────────────────────────────────
#  평가 지표: 각도 오차 (Mean Angular Error)
# ─────────────────────────────────────────────────────────────

def mean_angular_error_deg(pred: torch.Tensor, target: torch.Tensor) -> float:
    """예측 시선각도와 실제 시선각도의 평균 각도 오차를 계산합니다.
    
    Args:
        pred   : (B, 2) [yaw, pitch] 예측값 (radians)
        target : (B, 2) [yaw, pitch] 정답값 (radians)
    
    Returns:
        float: 평균 각도 오차 (degrees)
    
    원리: 시선벡터를 3D로 복원한 뒤 내적(dot product)으로 두 벡터의 각도 계산
    """
    def angles_to_vec(angles):
        # (B, 2) [yaw, pitch] → (B, 3) 3D 단위벡터
        yaw   = angles[:, 0]
        pitch = angles[:, 1]
        x = torch.sin(yaw) * torch.cos(pitch)
        y = -torch.sin(pitch)
        z = -torch.cos(yaw) * torch.cos(pitch)
        return torch.stack([x, y, z], dim=1)  # (B, 3)
    
    pred_vec   = angles_to_vec(pred.detach().cpu())
    target_vec = angles_to_vec(target.detach().cpu())
    
    # 내적 (코사인 유사도)
    cos_sim = (pred_vec * target_vec).sum(dim=1).clamp(-1.0, 1.0)
    # 각도 오차 (라디안 → 도)
    angular_err = torch.acos(cos_sim) * (180.0 / math.pi)
    
    return angular_err.mean().item()


print('\n📐 평가 지표: Mean Angular Error (MAE in degrees)')
print('  → 사람 눈 추적 정밀도 기준: ~3° 이하가 실용적 수준')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  학습/검증 함수 정의
# ─────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, device):
    """한 에포크 학습 수행.
    
    Returns:
        avg_loss: 에포크 평균 손실
        avg_angle_err: 에포크 평균 각도 오차 (degrees)
    """
    model.train()  # 학습 모드: Dropout 활성화, BatchNorm 학습 상태
    total_loss = 0.0
    total_angle_err = 0.0
    n_batches = len(loader)
    
    for images, labels in loader:
        images = images.to(device, non_blocking=True)  # GPU로 비동기 전송
        labels = labels.to(device, non_blocking=True)
        
        # 순전파 (Forward Pass)
        optimizer.zero_grad()        # 이전 배치의 gradient 초기화
        preds = model(images)        # 예측: (B, 2)
        
        # 손실 계산
        loss = criterion(preds, labels)
        
        # 역전파 (Backward Pass)
        loss.backward()              # gradient 계산
        
        # Gradient Clipping: 폭발적 gradient 방지 (RNN에서 주로 쓰지만 안전 장치)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()             # 파라미터 업데이트
        
        total_loss      += loss.item()
        total_angle_err += mean_angular_error_deg(preds, labels)
    
    return total_loss / n_batches, total_angle_err / n_batches


@torch.no_grad()  # 검증 시 gradient 계산 없음 → 메모리/속도 최적화
def validate(model, loader, criterion, device):
    """검증 세트 평가."""
    model.eval()  # 평가 모드: Dropout 비활성화, BatchNorm 고정
    total_loss = 0.0
    total_angle_err = 0.0
    n_batches = len(loader)
    
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        preds = model(images)
        loss  = criterion(preds, labels)
        
        total_loss      += loss.item()
        total_angle_err += mean_angular_error_deg(preds, labels)
    
    return total_loss / n_batches, total_angle_err / n_batches


print('🔧 학습/검증 함수 정의 완료')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  MLflow 연동 메인 학습 루프
# ─────────────────────────────────────────────────────────────

# MLflow 추적 서버 설정 (로컬 파일 시스템에 저장)
mlflow.set_tracking_uri(f'file:///{str(MLFLOW_DIR).replace(chr(92), "/")}')
mlflow.set_experiment('MPIIGaze_GazeEstimation')  # 실험 이름

# 실험 결과 저장 폴더
GAZE_EXP_DIR = get_next_exp_dir(RUNS_DIR / 'gaze')
print(f'📁 시선 추정 실험 경로: {GAZE_EXP_DIR}')

# 학습 기록 저장용 리스트
history = {
    'train_loss':  [],
    'val_loss':    [],
    'train_angle': [],
    'val_angle':   [],
    'lr':          []
}

best_val_loss = float('inf')  # 최고 검증 손실 추적
best_epoch    = 0

# ── MLflow Run 시작 ───────────────────────────────────────────
with mlflow.start_run(run_name=f'GazeEstimator_{GAZE_EXP_DIR.name}') as run:
    
    # 하이퍼파라미터 기록 (실험 재현성을 위해 모두 기록)
    mlflow.log_params({
        'model':        'MobileNetV3-Small',
        'input_size':   GAZE_INPUT_SIZE,
        'output_dim':   GAZE_OUTPUT_DIM,
        'epochs':       GAZE_EPOCHS,
        'batch_size':   GAZE_BATCH,
        'lr_init':      GAZE_LR,
        'lr_step':      GAZE_LR_STEP,
        'lr_gamma':     GAZE_LR_GAMMA,
        'weight_decay': GAZE_WEIGHT_DECAY,
        'loss':         'SmoothL1Loss',
        'optimizer':    'Adam',
        'train_samples': n_train,
        'val_samples':   n_val,
        'device':        DEVICE,
    })
    
    print(f'\n🚀 시선 추정 모델 학습 시작')
    print(f'   MLflow Run ID: {run.info.run_id[:8]}...')
    print(f'   에포크: {GAZE_EPOCHS}, 배치: {GAZE_BATCH}, LR: {GAZE_LR}')
    print('-' * 65)
    print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val Loss":>10} | {"Train °":>8} | {"Val °":>8} | {"LR":>8}')
    print('-' * 65)
    
    for epoch in range(1, GAZE_EPOCHS + 1):
        
        # 학습
        train_loss, train_angle = train_one_epoch(
            gaze_model, train_loader, criterion, optimizer, DEVICE
        )
        
        # 검증
        val_loss, val_angle = validate(
            gaze_model, val_loader, criterion, DEVICE
        )
        
        # 학습률 스케줄러 업데이트
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        
        # 기록 저장
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_angle'].append(train_angle)
        history['val_angle'].append(val_angle)
        history['lr'].append(current_lr)
        
        # MLflow 지표 기록
        mlflow.log_metrics({
            'train_loss':        train_loss,
            'val_loss':          val_loss,
            'train_angle_err':   train_angle,
            'val_angle_err':     val_angle,
            'learning_rate':     current_lr,
        }, step=epoch)
        
        # 콘솔 출력
        print(f'{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | '
              f'{train_angle:>7.2f}° | {val_angle:>7.2f}° | {current_lr:>8.6f}')
        
        # Best 모델 저장 (검증 손실 기준)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch    = epoch
            
            best_weights_path = GAZE_EXP_DIR / 'best.pt'
            torch.save({
                'epoch':       epoch,
                'model_state': gaze_model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'val_loss':    val_loss,
                'val_angle':   val_angle,
                'config': {
                    'input_size': GAZE_INPUT_SIZE,
                    'output_dim': GAZE_OUTPUT_DIM,
                }
            }, best_weights_path)
            print(f'         💾 Best 모델 저장! (val_loss={val_loss:.4f}, angle={val_angle:.2f}°)')
        
        # 5에포크마다 체크포인트 저장
        if epoch % 5 == 0:
            ckpt_path = GAZE_EXP_DIR / f'checkpoint_ep{epoch:03d}.pt'
            torch.save(gaze_model.state_dict(), ckpt_path)
    
    print('-' * 65)
    print(f'\n✅ 학습 완료!')
    print(f'   Best Epoch:     {best_epoch}')
    print(f'   Best Val Loss:  {best_val_loss:.4f}')
    print(f'   Best Model:     {best_weights_path}')
    
    # MLflow에 최종 성능 기록
    mlflow.log_metrics({
        'best_val_loss':  best_val_loss,
        'best_epoch':     best_epoch,
    })
    
    # Best 모델을 MLflow artifact로 등록
    mlflow.pytorch.log_model(
        gaze_model,
        artifact_path='gaze_estimator',
        registered_model_name='GazeEstimator_MobileNetV3'
    )
    
    print(f'\n📊 MLflow 대시보드: mlflow ui --backend-store-uri {MLFLOW_DIR}')
    print(f'   Run ID: {run.info.run_id}')

## 8️⃣ 학습 결과 시각화 & 평가

In [ ]:
# ─────────────────────────────────────────────────────────────
#  학습 곡선 시각화
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# (1) 손실 곡선
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   'r-', label='Val Loss',   linewidth=2)
axes[0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.7, label=f'Best (ep{best_epoch})')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('SmoothL1 Loss', fontsize=12)
axes[0].set_title('손실 곡선 (Loss Curve)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# (2) 각도 오차 곡선
axes[1].plot(epochs_range, history['train_angle'], 'b-', label='Train MAE', linewidth=2)
axes[1].plot(epochs_range, history['val_angle'],   'r-', label='Val MAE',   linewidth=2)
axes[1].axhline(y=3.0, color='orange', linestyle=':', alpha=0.8, label='목표: 3°')
axes[1].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Mean Angular Error (°)', fontsize=12)
axes[1].set_title('각도 오차 (Angular Error)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# (3) 학습률 변화
axes[2].plot(epochs_range, history['lr'], 'g-', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('학습률 스케줄 (LR Schedule)', fontsize=12, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.suptitle('GazeEstimator (MobileNetV3-Small) 학습 결과', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = GAZE_EXP_DIR / 'training_curves.png'
plt.savefig(str(plot_path), dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ 학습 곡선 저장: {plot_path}')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Best 모델로 예측 시각화
# ─────────────────────────────────────────────────────────────

# Best 모델 가중치 로드
checkpoint = torch.load(str(GAZE_EXP_DIR / 'best.pt'), map_location=DEVICE)
gaze_model.load_state_dict(checkpoint['model_state'])
gaze_model.eval()

print(f'✅ Best 모델 로드 (Epoch {checkpoint["epoch"]}, '
      f'val_angle={checkpoint["val_angle"]:.2f}°)')

# 검증 세트 배치에서 샘플 예측
sample_images, sample_labels = next(iter(val_loader))
sample_images = sample_images.to(DEVICE)

with torch.no_grad():
    sample_preds = gaze_model(sample_images).cpu()

# 역정규화: 이미지를 시각화할 수 있도록 정규화 되돌리기
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

n_show = min(8, len(sample_images))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(n_show):
    # 이미지 역정규화
    img_vis = sample_images[i].cpu() * std + mean
    img_vis = img_vis.permute(1, 2, 0).clamp(0, 1).numpy()
    
    pred_yaw   = math.degrees(sample_preds[i, 0].item())
    pred_pitch = math.degrees(sample_preds[i, 1].item())
    true_yaw   = math.degrees(sample_labels[i, 0].item())
    true_pitch = math.degrees(sample_labels[i, 1].item())
    
    # 각도 오차 계산
    err = mean_angular_error_deg(
        sample_preds[i:i+1],
        sample_labels[i:i+1]
    )
    
    axes[i].imshow(img_vis[:, :, 0], cmap='gray')  # 회색조로 표시
    axes[i].set_title(
        f'GT: yaw={true_yaw:.1f}°, p={true_pitch:.1f}°\n'
        f'PR: yaw={pred_yaw:.1f}°, p={pred_pitch:.1f}°\n'
        f'Err: {err:.2f}°',
        fontsize=8,
        color='green' if err < 5 else 'red'
    )
    axes[i].axis('off')

plt.suptitle('시선 추정 예측 결과 (초록: 5° 이내, 빨강: 5° 이상)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(str(GAZE_EXP_DIR / 'predictions.png'), dpi=100, bbox_inches='tight')
plt.show()
print('✅ 예측 결과 저장 완료')

## 9️⃣ ONNX 내보내기 — 웹 브라우저 배포 준비

> **목표:** `onnxruntime-web`으로 브라우저에서 직접 실행  
> ONNX 변환 → 양자화(FP16) → 크기 및 속도 최종 확인

In [ ]:
# ─────────────────────────────────────────────────────────────
#  ONNX 내보내기
# ─────────────────────────────────────────────────────────────

import torch.onnx

onnx_path = GAZE_EXP_DIR / 'gaze_estimator.onnx'

# Best 모델이 이미 로드되어 있음 (평가 모드)
gaze_model.eval()

# ONNX 변환용 더미 입력 (배치 크기=1, 웹에서는 1장씩 처리)
dummy_input = torch.randn(1, 3, GAZE_INPUT_SIZE, GAZE_INPUT_SIZE).to(DEVICE)

torch.onnx.export(
    gaze_model,
    dummy_input,
    str(onnx_path),
    
    # 입출력 이름 지정 (JavaScript에서 참조할 때 사용)
    input_names  = ['eye_image'],    # 입력 텐서 이름
    output_names = ['gaze_angles'],  # 출력 텐서 이름 [yaw, pitch]
    
    # 동적 배치 크기 허용 (브라우저에서 배치 크기가 달라질 수 있음)
    dynamic_axes = {
        'eye_image':   {0: 'batch_size'},
        'gaze_angles': {0: 'batch_size'}
    },
    
    opset_version   = 17,    # ONNX opset 버전 (onnxruntime-web 지원 버전)
    export_params   = True,  # 학습된 가중치 포함
    do_constant_folding = True,  # 상수 연산 사전 계산 (최적화)
    verbose         = False,
)

onnx_size_mb = os.path.getsize(onnx_path) / 1e6
print(f'✅ ONNX 내보내기 완료')
print(f'   경로: {onnx_path}')
print(f'   크기: {onnx_size_mb:.2f} MB')

# ─────────────────────────────────────────────────────────────
#  ONNX 모델 검증 (PyTorch vs ONNX 출력 비교)
# ─────────────────────────────────────────────────────────────
try:
    import onnxruntime as ort
    
    sess = ort.InferenceSession(str(onnx_path))
    dummy_np = dummy_input.cpu().numpy()
    
    # ONNX 추론
    onnx_out = sess.run(['gaze_angles'], {'eye_image': dummy_np})[0]
    
    # PyTorch 추론
    with torch.no_grad():
        torch_out = gaze_model(dummy_input).cpu().numpy()
    
    max_diff = abs(onnx_out - torch_out).max()
    print(f'\n✅ ONNX 검증: PyTorch ↔ ONNX 최대 오차 = {max_diff:.2e}')
    if max_diff < 1e-4:
        print('   → 변환 정확도 이상 없음')
    else:
        print('   ⚠️ 오차가 큼, 재확인 필요')
        
except ImportError:
    print('\n💡 ONNX 검증: onnxruntime 미설치')
    print('   pip install onnxruntime 후 실행하세요')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  YOLOv8 ONNX 내보내기
# ─────────────────────────────────────────────────────────────

best_yolo_weights = YOLO_EXP_DIR / 'weights' / 'best.pt'
if best_yolo_weights.exists():
    yolo_best = YOLO(str(best_yolo_weights))
    
    # YOLOv8 내장 ONNX 내보내기
    yolo_onnx_path = yolo_best.export(
        format    = 'onnx',
        imgsz     = YOLO_IMGSZ,
        opset     = 17,
        simplify  = True,  # ONNX 모델 단순화 (속도 향상)
        dynamic   = True,  # 동적 배치 크기
    )
    
    yolo_onnx_size = os.path.getsize(yolo_onnx_path) / 1e6
    print(f'✅ YOLOv8 ONNX 내보내기 완료')
    print(f'   경로: {yolo_onnx_path}')
    print(f'   크기: {yolo_onnx_size:.2f} MB')
else:
    print('⚠️ YOLOv8 best.pt 없음. 먼저 YOLO 학습을 완료하세요.')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  최종 결과 요약
# ─────────────────────────────────────────────────────────────

print('=' * 65)
print(' 🎉 학습 완료 요약')
print('=' * 65)
print()
print('📦 생성된 파일:')
print(f'  YOLO 눈 탐지 모델:')
print(f'    {YOLO_EXP_DIR}/weights/best.pt')
print(f'    {YOLO_EXP_DIR}/weights/best.onnx  (웹 배포용)')
print()
print(f'  시선 추정 모델:')
print(f'    {GAZE_EXP_DIR}/best.pt            (PyTorch)')
print(f'    {GAZE_EXP_DIR}/gaze_estimator.onnx (웹 배포용)')
print(f'    {GAZE_EXP_DIR}/training_curves.png')
print(f'    {GAZE_EXP_DIR}/predictions.png')
print()
print('📊 성능 (best epoch):')
if history['val_angle']:
    best_angle = min(history['val_angle'])
    print(f'  Val Angular Error:  {best_angle:.2f}°')
    print(f'  Val Loss:           {best_val_loss:.4f}')
print()
print('🌐 다음 단계 — 웹 배포:')
print('  1. ONNX 모델을 웹 서버에 업로드')
print('  2. JavaScript에서 onnxruntime-web 로드:')
print('     const session = await ort.InferenceSession.create("gaze_estimator.onnx");')
print('  3. 카메라 프레임 → 눈 탐지(YOLO) → 시선 추정(GazeEstimator)')
print()
print('📈 MLflow 대시보드 실행:')
print(f'  mlflow ui --backend-store-uri {MLFLOW_DIR}')
print(f'  → http://localhost:5000')
print()
print('💙 ALS 환우분들에게 자유로운 소통을 선물하세요!')
print('=' * 65)